## Exploring the dataset

In [1]:
import json

rows = [json.loads(line) for line in open("data/train.jsonl", encoding="utf-8")]

print("number of lines:", len(rows))
rows[0]

number of lines: 478


{'id': 2,
 'text': 'De sålde deras hus förra året.',
 'score': 0.45,
 'label': 'poor',
 'error': "wrong pronoun form ('sitt' -> 'deras')",
 'group': 434}

In [2]:
# just putting everything in a dataframe
import pandas as pd

df = pd.DataFrame(rows)

df.head(10)

,id,text,score,label,error,group
0,2,De sålde deras hus förra året.,0.45,poor,wrong pronoun form ('sitt' -> 'deras'),434
1,4,Vi måste vänta för bussen i tjugo minuter.,0.40,poor,wrong preposition (vänta på),4
2,5,Trots att vädret var dåligt kom nästan alla ti...,0.95,good,NaN,5
3,6,Om symtomen inte går över inom en vecka bör du...,0.95,good,NaN,6
4,8,Förra året reste vi till Japan.,0.95,good,NaN,8
5,9,Företaget har anställde tjugo nya medarbetarer...,0.20,poor,wrong verb form ('anställt' -> 'anställde'); w...,141
6,10,"Efter jobbet gick jag till gymmet, men det var...",0.95,good,NaN,10
7,11,Jag bor på Sverige.,0.40,poor,wrong preposition with country (i Sverige),11
8,12,Hon köpte en ny vinter jacka.,0.45,poor,compound written as two words ('vinterjacka' -...,423
9,13,Ett barn satt på bänken och grät.,0.95,good,NaN,13


In [3]:
df.groupby("label")["score"].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
good,239.0,0.953138,0.021996,0.8,0.95,0.95,0.95,1.00
poor,239.0,0.350209,0.100889,0.0,0.30,0.40,0.40,0.55


In [4]:
df["score"].round(2).value_counts().sort_index()

score
0.00      1
0.05      4
0.10      4
0.15      7
0.20     20
0.25     16
0.30      9
0.35     54
0.40     73
0.45     47
0.50      2
0.55      2
0.80      1
0.85      1
0.90      7
0.95    203
1.00     27
Name: count, dtype: int64

In [5]:
# errors
poor = df[df["label"] == "poor"].copy() # only poor

for e in poor["error"].head(10):
    print(repr(e))

"wrong pronoun form ('sitt' -> 'deras')"
'wrong preposition (vänta på)'
"wrong verb form ('anställt' -> 'anställde'); wrong plural form ('medarbetare' -> 'medarbetarer')"
'wrong preposition with country (i Sverige)'
"compound written as two words ('vinterjacka' -> 'vinter jacka')"
'missing definite (modellen); compound split; agreement (själva)'
"compound written as two words ('tandborsten' -> 'tand borsten')"
'V2 violation; gender (ett gemensamt avtal); negation placement in subordinate clause'
"adverb/negation placement ('har fortfarande inte' -> 'har inte fortfarande')"
'wrong gender (ett förslag)'


In [6]:
poor["n_errors"] = poor["error"].apply(lambda e: len([p for p in e.split(";") if p.strip()]))

In [7]:
poor[["error", "n_errors"]].head(10)

,error,n_errors
0,wrong pronoun form ('sitt' -> 'deras'),1
1,wrong preposition (vänta på),1
5,wrong verb form ('anställt' -> 'anställde'); w...,2
7,wrong preposition with country (i Sverige),1
8,compound written as two words ('vinterjacka' -...,1
10,missing definite (modellen); compound split; a...,3
13,compound written as two words ('tandborsten' -...,1
15,V2 violation; gender (ett gemensamt avtal); ne...,3
16,adverb/negation placement ('har fortfarande in...,1
21,wrong gender (ett förslag),1


In [8]:
poor.groupby("n_errors")["score"].agg(["count", "mean", "min", "max"])

,count,mean,min,max
n_errors,,,,
1,184,0.392663,0.00,0.55
2,47,0.227660,0.10,0.40
3,8,0.093750,0.05,0.15


so the score depends on the number of errors

## Expanding the dataset for fine-tuning

we have only 478 examples which is not enough to fine-tune BERT, so I will use sentences in DaLAJ, it is also in huggingface

In [9]:
from datasets import load_dataset

dalaj = load_dataset("mteb/DalajClassification", split="train")

dalaj_df = dalaj.to_pandas()

/home/gusvahaye@GU.GU.SE/Swedish-Grammaticality-thesis/venv/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
dalaj_df.head(10)

,text,label
0,Det samma gäller inte för en kommunal lägenhet...,1
1,Några tider kan vi titta och lyssna på hur det...,1
2,Bonus son har fått ett akvarium i sitt rum som...,1
3,efterditt kan du träffa vänner som kommer från...,1
4,I det här fallet fungerar kursen som en hjälp ...,1
5,Andravärldskriget ligger i det förflutna och h...,1
6,Därför att detta gäller för de flesta nya arbe...,1
7,"Tillslut vill jag säga , som jag sagt till min...",1
8,Flera människor anser att det är rätt – andra ...,1
9,Hat kärleken vänder sig med en stark vilja av ...,1


label 0 = correct  
label 1 = incorrect

In [11]:
dalaj_df["label"].value_counts()

label
1    3841
0    3841
Name: count, dtype: int64

In [12]:
import re
def fix_spacing(text):
    return re.sub(r"\s+([.,!?:,])", r"\1", text)

dalaj_df["text"] = dalaj_df["text"].apply(fix_spacing)

In [13]:
dalaj_df["text"].head(5).to_list()

['Det samma gäller inte för en kommunal lägenhet i D-hemland, där vi tar varmvatten från en gaskokare.',
 'Några tider kan vi titta och lyssna på hur det funkar.',
 'Bonus son har fått ett akvarium i sitt rum som han är jättestolt över.',
 'efterditt kan du träffa vänner som kommer från olika länder.',
 'I det här fallet fungerar kursen som en hjälp till elever.']

In [14]:
def label_to_score(label):
    if label == 0:
        return 0.95 # correct
    else:
        return 0.4
    
dalaj_df["score"] = dalaj_df["label"].apply(label_to_score)
dalaj_df.head(-10)

,text,label,score
0,Det samma gäller inte för en kommunal lägenhet...,1,0.40
1,Några tider kan vi titta och lyssna på hur det...,1,0.40
2,Bonus son har fått ett akvarium i sitt rum som...,1,0.40
3,efterditt kan du träffa vänner som kommer från...,1,0.40
4,I det här fallet fungerar kursen som en hjälp ...,1,0.40
...,...,...,...
7667,I boken berättar hon om hur pojken föddes med ...,0,0.95
7668,I boken berättar hon om hur pojken föddes med ...,0,0.95
7669,Alex är en pojke som föddes med hundöron och h...,0,0.95
7670,Jag valde att beskriva Alex eftersom han är de...,0,0.95


In [15]:
dalaj_df["score"].value_counts()

score
0.40    3841
0.95    3841
Name: count, dtype: int64

In [16]:
toy_small = df[["text", "score"]]
dalaj_small = dalaj_df[["text", "score"]]

print(toy_small, dalaj_small)

                                                  text  score
0                       De sålde deras hus förra året.   0.45
1           Vi måste vänta för bussen i tjugo minuter.   0.40
2    Trots att vädret var dåligt kom nästan alla ti...   0.95
3    Om symtomen inte går över inom en vecka bör du...   0.95
4                      Förra året reste vi till Japan.   0.95
..                                                 ...    ...
473              Den sista bussen går kvart över tolv.   0.95
474                               Han har två brodrar.   0.40
475                           Han är bra på matematik.   0.95
476                       Han tvättade hans bil i går.   0.45
477                       Brevet var skriven för hand.   0.40

[478 rows x 2 columns]                                                    text  score
0     Det samma gäller inte för en kommunal lägenhet...   0.40
1     Några tider kan vi titta och lyssna på hur det...   0.40
2     Bonus son har fått ett akvarium i sit

In [17]:
train_df = pd.concat([dalaj_small, toy_small], ignore_index=True)
train_df.head(10)

,text,score
0,Det samma gäller inte för en kommunal lägenhet...,0.4
1,Några tider kan vi titta och lyssna på hur det...,0.4
2,Bonus son har fått ett akvarium i sitt rum som...,0.4
3,efterditt kan du träffa vänner som kommer från...,0.4
4,I det här fallet fungerar kursen som en hjälp ...,0.4
5,Andravärldskriget ligger i det förflutna och h...,0.4
6,Därför att detta gäller för de flesta nya arbe...,0.4
7,"Tillslut vill jag säga, som jag sagt till mina...",0.4
8,Flera människor anser att det är rätt – andra ...,0.4
9,Hat kärleken vänder sig med en stark vilja av ...,0.4


In [18]:
len(train_df)

8160

In [19]:
train_df["score"].round(1).value_counts()

score
1.0    4071
0.4    4015
0.2      43
0.3       9
0.9       7
0.0       5
0.1       4
0.6       2
0.5       2
0.8       2
Name: count, dtype: int64

In [20]:
train_df.tail(3)

,text,score
8157,Han är bra på matematik.,0.95
8158,Han tvättade hans bil i går.,0.45
8159,Brevet var skriven för hand.,0.40


In [21]:
# shuffle
train_df.sample(frac=1, random_state=42)

,text,score
4342,Jag tycker att hon var kort och tjock med stor...,0.40
1203,Språkvården arbetar aktivt för att nynorskan o...,0.95
7133,Han öppnar dörren lite grann för att checka in...,0.95
1758,Hon tror att killen från ett annat land är hen...,0.95
927,Man måste börja ett nytt liv när man flyttar t...,0.40
...,...,...
5226,När jag gifte mig fick jag hjälp av min mamma.,0.95
5390,Idag lever vi i en värld dominerad av teknolog...,0.95
860,I början ringde han till mig varje kväll efter...,0.40
7603,En dag gick Sofia och Fatima hem till Sofia fö...,0.95


In [22]:
train_df.to_json("data/my_train.jsonl", orient="records", lines=True, force_ascii=False)

adding ScaLA

In [25]:
scala = load_dataset("mteb/scala_sv_classification", split="full_train")
print(scala)
print(scala[0])
print(scala[1])

Dataset({
    features: ['text', 'corruption_type', 'label'],
    num_rows: 7446
})
{'text': 'Det framgår, menar man, av Jesu förkunnelse apostlarnas och förmaningar i Nya Testamentet, utan att vara direkt utsagt.', 'corruption_type': 'flip_neighbours', 'label': 'incorrect'}
{'text': 'Den period, som är den arbetsammaste men samtidigt den mest levande, betydelsefull för personlighetsutvecklingen både hos de vuxna och barnen, den period i livet när känslokontakten barn-föräldrar skapas, den perioden måste värderas särskilt högt.', 'corruption_type': None, 'label': 'correct'}


In [26]:
scala_df = scala.to_pandas()
scala_df.head(10)

,text,corruption_type,label
0,"Det framgår, menar man, av Jesu förkunnelse ap...",flip_neighbours,incorrect
1,"Den period, som är den arbetsammaste men samti...",NaN,correct
2,Omkring året börjar en del barn ta de första s...,delete,incorrect
3,"Dessa orter, där sålunda postnumret bestäms av...",flip_neighbours,incorrect
4,Dessa men länder - där tre fjärdedelar av jord...,flip_neighbours,incorrect
5,I går kunde SDS berätta om ett skelettfynd som...,NaN,correct
6,Men äktenskapets värde kan inte till begränsas...,flip_neighbours,incorrect
7,Det finns inga daghem och att få en dagmamma ä...,NaN,correct
8,Belopp som slutar på 1 eller 2 öre ska avrunda...,delete,incorrect
9,"Den ena av de dämpande, rogivande medlen.",delete,incorrect


In [27]:
scala_df["label"].value_counts()

label
incorrect    3723
correct      3723
Name: count, dtype: int64

In [28]:
def labelstr_to_score(label):
    if label == "correct":
        return 0.95
    else:
        return 0.4
    
scala_df["score"] = scala_df["label"].apply(labelstr_to_score)
scala_df.head(19)

,text,corruption_type,label,score
0,"Det framgår, menar man, av Jesu förkunnelse ap...",flip_neighbours,incorrect,0.40
1,"Den period, som är den arbetsammaste men samti...",NaN,correct,0.95
2,Omkring året börjar en del barn ta de första s...,delete,incorrect,0.40
3,"Dessa orter, där sålunda postnumret bestäms av...",flip_neighbours,incorrect,0.40
4,Dessa men länder - där tre fjärdedelar av jord...,flip_neighbours,incorrect,0.40
5,I går kunde SDS berätta om ett skelettfynd som...,NaN,correct,0.95
6,Men äktenskapets värde kan inte till begränsas...,flip_neighbours,incorrect,0.40
7,Det finns inga daghem och att få en dagmamma ä...,NaN,correct,0.95
8,Belopp som slutar på 1 eller 2 öre ska avrunda...,delete,incorrect,0.40
9,"Den ena av de dämpande, rogivande medlen.",delete,incorrect,0.40


In [29]:
scala_small = scala_df[["text", "score"]]
scala_small

,text,score
0,"Det framgår, menar man, av Jesu förkunnelse ap...",0.40
1,"Den period, som är den arbetsammaste men samti...",0.95
2,Omkring året börjar en del barn ta de första s...,0.40
3,"Dessa orter, där sålunda postnumret bestäms av...",0.40
4,Dessa men länder - där tre fjärdedelar av jord...,0.40
...,...,...
7441,Det går knappast att göra en cellplast så mjuk...,0.95
7442,Eftersom alkohol och många lugnande medel och ...,0.95
7443,Den allmänna pensionen är av två slag: folkpen...,0.95
7444,i Sverige utgörs till stor del av ungdomsbrott...,0.40


In [31]:
train_df_final = pd.concat([train_df, scala_small], ignore_index=True)

In [32]:
len(train_df_final)

15606

In [33]:
train_df_final["score"].round(1).value_counts()

score
1.0    7794
0.4    7738
0.2      43
0.3       9
0.9       7
0.0       5
0.1       4
0.6       2
0.5       2
0.8       2
Name: count, dtype: int64

In [34]:
train_df_final.sample(frac=1, random_state=42)

,text,score
14404,Resistensförädling mot bl.a. svampsjukdomar ho...,0.95
6070,Det finns nackdelar med betalning svart för en...,0.40
8950,"I praktiken, har invänt, varierar olika familj...",0.40
12266,De invandrarrikaste trakterna är Mälarområdet ...,0.40
14809,Tappa ur det gamla kylvattnet.,0.95
...,...,...
5191,Jag är trygg med att sitta på bussen för jag b...,0.95
13418,Den gamla samhällsordningen byggde på auktorit...,0.95
5390,Idag lever vi i en värld dominerad av teknolog...,0.95
860,I början ringde han till mig varje kväll efter...,0.40


In [35]:
train_df_final.to_json("data/my_train.jsonl", orient="records", lines=True, force_ascii=False)